In [ ]:
import torch
from darts.models import ARIMA
from darts.utils.missing_values import extract_subseries

from aare.AareDataset import AareDataset
from aare.evaluation.evaluation import evaluate_model
from aare.evaluation.forecast import Forecast
from aare.evaluation.forecast_samples import ForecastSamples
from aare.evaluation.metrics import Metrics
from aare.params import read_params
from aare.preparation import (
    prepare_ts,
)
from aare.remote_existenz_store import RemoteExistenzStore
from aare.utils import get_context_len

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()
store = RemoteExistenzStore()
ds = AareDataset.from_conf()

In [ ]:
train = prepare_ts(ds.get_train())
train

In [ ]:
train_subs = extract_subseries(train, mode="any")
train_l = sorted(train_subs, key=len, reverse=True)[0]
train_l

In [ ]:
val = prepare_ts(ds.get_val())
val

In [ ]:
val_subs = extract_subseries(val, mode="any")
val_l = sorted(val_subs, key=len, reverse=True)[0]
val_l

# ARIMA(X) (with just air temp)

Reasons I think this might work:

1. The water temperatures seems to follow an AR(1) process and is stationary after first difference
2. The air temperature (forecast), which is probably our most dominant covariate/predictor, captures the same seasonality as the water temperature (daily and yearly), so we don't need to worry about seasonality. This is an especially important point because ARIMA cannot handle multiple seasonalities (should use TBATS instead for example).

Reasons this might not perform very well:

1. The relationship between water and air temperature is non-linear at low and high temperatures (DOI 10.1029/98WR01877). Could use some non-linear transformations of the air temp as covariates to help with this.
2. We already know lag 1 (hour) of the air temperature is the best lag (highest correlation), so there is a seasonality discrepancy of 1 hour
3. There are other factors that influence the water temperature and the relationships and interactions are probably much more complex than ARIMAX can model.


In [ ]:
# arima = ARIMA(p=1, d=1, q=0, add_encoders={'cyclic': {'future': ['hour']}})
# just as a first
arima = ARIMA(p=1, d=1, q=0)
arima

In [ ]:
arima.fit(train)

In [ ]:
validation_params = params["validation"]
stride = validation_params["stride"]
min_lookback_hours = validation_params["min_lookback_hours"]
forecast_horizon = params["general"]["forecast_horizon"]

In [ ]:
# THIS TOOK 24m to run!!! TODO implement parallelized evaluation and do it again :) also importing a different func now cuz rename
# ALSO TODO: What actually happens when you evaluate a local forecasting model like this? The historical forecast method says that
# retrain=False is only supported for global models, but is it?
# LMAO the docs are wrong. There are TransferableFutureCovariatesLocalForecastingModel (like our ARIMA),
# which interally set _supports_non_retrainable_historical_forecasts to true, so actually some local models
# work too, hooray. Could even submit an issue. Ps. the public property seems to be supports_transferrable_series_prediction,
# but they are only similar semantically and not bound together programmatically (not sure why).
(
    metrics,
    (last_prediction, last_prediction_m),
    (best_prediction, best_prediction_m),
    (worst_prediction, worst_prediction_m),
) = evaluate_model(arima, val_subs, stride, forecast_horizon)

In [ ]:
# Oh and another TODO!
# Analyze how to get the correct context length of a model, maybe you need special cases for some classes like ARIMA. we care about the predict-time context-length.
# https://github.com/unit8co/darts/blob/42776790183bbe42411fc2dba3e1e8416f9263e8/darts/models/forecasting/forecasting_model.py#L3360
# ARIMA just hard-codes 30 as their min train length, which cannot be the actual context len: https://github.com/unit8co/darts/blob/42776790183bbe42411fc2dba3e1e8416f9263e8/darts/models/forecasting/arima.py#L233
# It seems that the models that we can work with are "transferrable" models, and all global models are tranferrable, but only few
# local ones are.
# TODO figure out if there's a common base class we could use, but probably not
# TODO also add a verbose flag to the evaluation
# ALSO: ARIMA supports probabilistic forecasts and we love that, so make sure to also add that to the eval pipeline
# and think about where to put the number of samples [params] (i.e. is that a general setting for all models, one per model, etc).

# Searching the Darts codebase, the following models support transferrable series prediction:
# - GlobalForecastingModel and all derivatives
# - TransferableFutureCovariateLocalForecastingModel and all derivatives, which currently are ARIMA, VARIMA and KalmanForecaster

# Similarly, the following models support non retrainable historical forecasts
# - GlobalForecastingModel and all derivatives
# - TransferableFutureCovariateLocalForecastingModel and all derivatives
# - Global ensemble models

# So only ensembles are different, but interestingly, EnsembleModel is a global model, so a global model always supports transferrable series prediction,
# but it only supports non-retrainable historical forecasts if all models in the ensemble are global models.

In [ ]:
metrics

In [ ]:
get_context_len(arima)

In [ ]:
arima.supports_transferrable_series_prediction
arima._supports_non_retrainable_historical_forecasts

In [ ]:
lookback_hours = max(get_context_len(arima), min_lookback_hours)

last_forecast = Forecast(val, last_prediction, lookback_hours, Metrics.from_ndarray(last_prediction_m))
best_forecast = Forecast(val, best_prediction, lookback_hours, Metrics.from_ndarray(best_prediction_m))
worst_forecast = Forecast(val, worst_prediction, lookback_hours, Metrics.from_ndarray(worst_prediction_m))
sample = ForecastSamples(last_forecast, best_forecast, worst_forecast)

In [ ]:
title = "ARIMA"
fig, axes = plt.subplot_mosaic("AA;BC")

last_forecast.plot(title + " (last)", ax=axes["A"])
best_forecast.plot(title + " (best)", ax=axes["B"])
worst_forecast.plot(title + " (worst)", ax=axes["C"])